In [1]:
from fastapi import FastAPI
import joblib
import pandas as pd
from pydantic import BaseModel
import uvicorn
import asyncio
import nest_asyncio

In [2]:
def lead_time_fill(df):

    df['Lead_Time'] = df['Lead_Time'].fillna(df['Lead_Time'].mean())

    return df

In [3]:
def outliers_capping(df):

   for col in ['Quantity','Defective_Units']:
      
      q3 = df[col].quantile(0.75)
      q1 = df[col].quantile(0.25)
      iqr = q3 - q1
      lb = q1 - 1.5*iqr
      ub = q3 + 1.5*iqr

      df[col] = df[col].clip(lower=lb, upper=ub)

      return df

In [ ]:
app = FastAPI(title='Defective Units Prediction API')

pipeline = joblib.load('fitted_pipeline.pkl')
model = joblib.load('model.pkl')

class ProcurementInput(BaseModel):
    Supplier: str
    Item_Category: str
    Quantity: float
    Compliance: str
    Lead_Time: float
    Negotiated_vs_Unit_Price: float

@app.get("/")
def home():
    return {"message": "Defective Units Prediction API is running"}


@app.post("/predict")
def predict(data: ProcurementInput):

    input_df = pd.DataFrame([data.model_dump()])
    input_transformed = pipeline.transform(input_df)
    prediction = model.predict(input_transformed)

    return {"Predicted_Defective_Units": float(prediction[0])}

#nest_asyncio.apply()
#uvicorn.run(app, host="127.0.0.1", port=8000)

async def run():
    config = uvicorn.Config(app, host="127.0.0.1", port=8000)
    server = uvicorn.Server(config)
    await server.serve()

asyncio.create_task(run())

<Task pending name='Task-4' coro=<run() running at C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10412\3362549374.py:36>>

INFO:     Started server process [10412]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52371 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:52371 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:52371 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:52372 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:52388 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:52390 - "POST /predict HTTP/1.1" 200 OK
